# REC Financial Feasibility Analysis
## Renewable Energy Community – PV & Battery Optimizer

**Step-by-step guide for beginners**

This notebook analyzes four scenarios:
1. **Grid only** (baseline)
2. **PV + Grid**
3. **PV + Battery + Grid**
4. **Battery + Grid**

Then finds optimal system sizes for PV + Grid, PV + Battery + Grid, and Battery + Grid configurations.

---
## Step 1: Install dependencies (Colab: run this cell first)

If using **Google Colab**: Click "Runtime" → "Run all" or run each cell with Shift+Enter.

If using **local Jupyter**: Run `pip install -r requirements.txt` in terminal first.

In [ ]:
# Install packages (Colab usually has these; uncomment if needed)
# !pip install pandas openpyxl matplotlib seaborn -q

import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

print("✓ Libraries loaded")

---
## Step 2: Load your data files

**Option A – Google Colab:** Run the cell below to upload files from your computer.

**Option B – Local / Google Drive:** Set `USE_UPLOAD = False` and edit the file paths.

In [ ]:
USE_UPLOAD = True  # Set False if using file paths below

if USE_UPLOAD:
    from google.colab import files
    print("Upload CONSUMPTION CSV (e.g. Houly_Data_kWh.csv):")
    uploaded = files.upload()
    cons_path = list(uploaded.keys())[0]
    print("\nUpload PV PRODUCTION CSV (e.g. Timeseries_...1kWp....csv):")
    uploaded2 = files.upload()
    prod_path = list(uploaded2.keys())[0]
else:
    # Edit these paths for local / Drive
    cons_path = "/content/drive/MyDrive/Houly_Data_kWh.csv"
    prod_path = "/content/drive/MyDrive/Timeseries_52.141_-10.269_SA3_1kWp_crystSi_14_35deg_0deg_2020_2020.csv"

print(f"Consumption: {cons_path}")
print(f"Production:  {prod_path}")

---
## Step 3: Parse and align data

- **Consumption:** Uses `Final_Community_Sum` (kWh per hour)
- **PV profile:** From Timeseries `P` column – Wh for 1 kWp → kWh/kWp. Timestamps like `01:11` treated as hour `1`.

In [ ]:
# Load consumption
df_cons = pd.read_csv(cons_path)
# Normalize date: handle "1:11" -> "1:00"
df_cons['date'] = pd.to_datetime(df_cons['date'], format='%d/%m/%Y %H:%M', errors='coerce')
df_cons['date'] = df_cons['date'].dt.floor('h')  # round to hour
df_cons = df_cons.dropna(subset=['date'])
consumption_col = 'Final_Community_Sum' if 'Final_Community_Sum' in df_cons.columns else df_cons.columns[1]
df_cons = df_cons[['date', consumption_col]].rename(columns={consumption_col: 'consumption'})
df_cons = df_cons[df_cons['date'].dt.year == 2020].reset_index(drop=True)  # use 2020 only

# Load PV production (Timeseries format)
# Find the header row (contains "time" and "P") - PVGIS files vary
try:
    with open(prod_path, 'r', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if 'time' in line.lower() and ('P' in line or 'G(i)' in line):
                header_row = i
                break
        else:
            header_row = 10
except Exception:
    header_row = 10

df_pv_raw = pd.read_csv(prod_path, skiprows=header_row)
df_pv_raw.columns = [c.strip() for c in df_pv_raw.columns]
# Parse time: 20200101:0011 -> 2020-01-01 00:00 (treat HH11 as hour HH)
def parse_pv_time(s):
    s = str(s).strip()
    # Only parse valid format: YYYYMMDD:HHmm (8 digits, colon, 4 digits)
    if len(s) < 12 or not s[:8].isdigit() or s[8] != ':' or not s[9:11].isdigit():
        return pd.NaT
    date_part = s[:8]
    hour_part = s[9:11]
    return pd.Timestamp(f"{date_part[:4]}-{date_part[4:6]}-{date_part[6:8]} {int(hour_part):02d}:00:00")

time_col = 'time' if 'time' in df_pv_raw.columns else df_pv_raw.columns[0]
df_pv_raw['date'] = df_pv_raw[time_col].apply(parse_pv_time)
df_pv_raw = df_pv_raw.dropna(subset=['date'])  # drop header/metadata rows
# P = Wh for 1 kWp -> kWh per kWp per hour
p_col = 'P' if 'P' in df_pv_raw.columns else df_pv_raw.columns[1]
df_pv_raw['pv_per_kwp'] = pd.to_numeric(df_pv_raw[p_col], errors='coerce').fillna(0) / 1000.0
df_pv = df_pv_raw[['date', 'pv_per_kwp']].copy()

# Merge on date
df = df_cons.merge(df_pv, on='date', how='left').reset_index(drop=True)
df['pv_per_kwp'] = df['pv_per_kwp'].fillna(0)

print(f"Loaded {len(df)} hourly records for 2020")
print(f"Total consumption: {df['consumption'].sum():.0f} kWh")
print(f"PV profile (1 kWp) total: {df['pv_per_kwp'].sum():.2f} kWh/kWp/year")
df.head()

---
## Step 4: Tariff configuration

From your `tariffs.xlsx`:
- **Standard:** day/peak/night (cents → €)
- **Weekend Saver:** weekday + weekend rates
- **Flat:** single rate 24/7
- **Export:** €0.1886/kWh

**Time bands:** Peak 17–19, Night 23–8, Day 8–17 & 19–23

In [ ]:
# Tariffs (€/kWh) - from tariffs.xlsx, converted from cents
TARIFFS = {
    'standard': {'day': 0.32067, 'peak': 0.36526, 'night': 0.2076},
    'weekend': {
        'weekday': {'day': 0.3268, 'peak': 0.38095, 'night': 0.2476},
        'weekend': {'day': 0.22205, 'peak': 0.23265, 'night': 0.19025},
    },
    'flat': 0.29966,
}
EXPORT_RATE = 0.1886  # €/kWh
CO2_FACTOR = 0.35    # kg CO2/kWh (350 g)

def get_tariff(t, tariff_type='standard'):
    hour = t.hour
    is_weekend = t.dayofweek >= 5  # Sat=5, Sun=6
    if tariff_type == 'flat':
        return TARIFFS['flat']
    if 17 <= hour < 19:
        band = 'peak'
    elif hour >= 23 or hour < 8:
        band = 'night'
    else:
        band = 'day'
    if tariff_type == 'weekend':
        key = 'weekend' if is_weekend else 'weekday'
        return TARIFFS['weekend'][key][band]
    return TARIFFS['standard'][band]

# Add tariff to each row
for tt in ['standard', 'weekend', 'flat']:
    df[f'tariff_{tt}'] = df['date'].apply(lambda t, tt=tt: get_tariff(t, tt))
print("✓ Tariffs applied")

---
## Step 5: Scenario simulation engine

Implements the four scenarios for a given PV size (kWp) and battery size (kWh).

In [ ]:
# Battery parameters
BATT_EFF = 0.9           # round-trip efficiency (sqrt for charge/discharge)
BATT_DOD = 0.9           # usable depth of discharge
BATT_INIT_SOC = 0.0      # initial state of charge (fraction)

def run_scenario_grid_only(df, tariff_col='tariff_standard'):
    """Scenario 1: All from grid"""
    df = df.copy()
    df['grid_import'] = df['consumption']
    df['feed_in'] = 0.0
    return df

def run_scenario_pv_grid(df, pv_kwp, tariff_col='tariff_standard'):
    """Scenario 2: PV + Grid (no battery)"""
    df = df.copy()
    pv_prod = df['pv_per_kwp'] * pv_kwp
    df['self_cons'] = np.minimum(pv_prod, df['consumption'])
    df['grid_import'] = np.maximum(df['consumption'] - pv_prod, 0)
    df['feed_in'] = np.maximum(pv_prod - df['consumption'], 0)
    return df

def run_scenario_battery_grid(df, batt_kwh, tariff_col='tariff_standard'):
    """Scenario 4: Battery + Grid (no PV). Charge at night, discharge at peak."""
    df = df.copy()
    df['pv_prod'] = 0.0
    eff = np.sqrt(BATT_EFF)
    max_power = batt_kwh * 0.5  # 0.5C
    usable = batt_kwh * BATT_DOD
    soc = batt_kwh * BATT_INIT_SOC
    grid_imp, feed_in, batt_ch, batt_dch = [], [], [], []
    for _, row in df.iterrows():
        cons = row['consumption']
        tariff = row[tariff_col]
        hour = row['date'].hour
        is_night = hour >= 23 or hour < 8
        is_peak = 17 <= hour < 19
        ch_from_grid = 0.0
        dch = 0.0
        if is_night:
            space = usable - soc
            ch_from_grid = min(space, max_power) / eff
            soc += ch_from_grid * eff
        elif not is_night and soc > 0:
            # Discharge during day and peak (tariff > night) to maximize arbitrage
            dch = min(soc, cons, max_power)
            soc -= dch
        grid_needed = max(0, cons - dch)
        grid_imp.append(grid_needed + ch_from_grid)
        feed_in.append(0.0)
        batt_ch.append(ch_from_grid)
        batt_dch.append(dch)
    df['grid_import'] = grid_imp
    df['feed_in'] = feed_in
    return df

def run_scenario_pv_battery_grid(df, pv_kwp, batt_kwh, tariff_col='tariff_standard'):
    """Scenario 3: PV + Battery + Grid. Battery charges from PV surplus and grid at night."""
    df = df.copy()
    pv_prod = df['pv_per_kwp'] * pv_kwp
    eff = np.sqrt(BATT_EFF)
    max_power = batt_kwh * 0.5
    usable = batt_kwh * BATT_DOD
    soc = batt_kwh * BATT_INIT_SOC
    grid_imp, feed_in = [], []
    for idx, (_, row) in enumerate(df.iterrows()):
        cons = row['consumption']
        pv = pv_prod.iloc[idx]
        tariff = row[tariff_col]
        hour = row['date'].hour
        is_night = hour >= 23 or hour < 8
        is_peak = 17 <= hour < 19
        # 1. Self-consumption from PV
        pv_to_load = min(pv, cons)
        pv_surplus = pv - pv_to_load
        cons_remaining = cons - pv_to_load
        # 2. Charge battery from PV surplus
        ch_from_pv = min(pv_surplus, (usable - soc) / eff, max_power / eff)
        soc += ch_from_pv * eff
        pv_surplus -= ch_from_pv
        # 3. Charge from grid at night if space
        ch_from_grid = 0.0
        if is_night and soc < usable:
            ch_from_grid = min((usable - soc) / eff, max_power / eff)
            soc += ch_from_grid * eff
        # 4. Discharge when load remains and tariff is high (peak or day)
        dch = 0.0
        if cons_remaining > 0 and soc > 0 and (is_peak or not is_night):
            dch = min(soc, cons_remaining, max_power)
            soc -= dch
            cons_remaining -= dch
        grid_imp.append(cons_remaining + ch_from_grid)
        feed_in.append(max(0, pv_surplus))
    df['grid_import'] = grid_imp
    df['feed_in'] = feed_in
    return df

print("✓ Scenario engine ready")

---
## Step 6: Run four-scenario comparison

Pick a reference PV and battery size (e.g. 50 kWp, 100 kWh) and compare all four scenarios across all three tariffs.

In [ ]:
REF_PV_KWP = 50    # Reference PV size for comparison
REF_BATT_KWH = 100 # Reference battery size

def calc_annual_metrics(d, tariff_col='tariff_standard'):
    cost = (d['grid_import'] * d[tariff_col]).sum() - (d['feed_in'] * EXPORT_RATE).sum()
    co2 = (d['grid_import'] * CO2_FACTOR).sum()
    return {'cost': cost, 'co2_kg': co2}

results = []
for tariff_name, tariff_col in [('Standard', 'tariff_standard'), ('Weekend Saver', 'tariff_weekend'), ('Flat', 'tariff_flat')]:
    # Scenario 1: Grid only
    d1 = run_scenario_grid_only(df, tariff_col)
    m1 = calc_annual_metrics(d1, tariff_col)
    m1['scenario'] = 'Grid only'; m1['tariff'] = tariff_name
    results.append(m1)
    # Scenario 2: PV + Grid
    d2 = run_scenario_pv_grid(df, REF_PV_KWP, tariff_col)
    m2 = calc_annual_metrics(d2, tariff_col)
    m2['scenario'] = 'PV + Grid'; m2['tariff'] = tariff_name
    results.append(m2)
    # Scenario 3: PV + Battery + Grid
    d3 = run_scenario_pv_battery_grid(df, REF_PV_KWP, REF_BATT_KWH, tariff_col)
    m3 = calc_annual_metrics(d3, tariff_col)
    m3['scenario'] = 'PV + Battery + Grid'; m3['tariff'] = tariff_name
    results.append(m3)
    # Scenario 4: Battery + Grid
    d4 = run_scenario_battery_grid(df, REF_BATT_KWH, tariff_col)
    m4 = calc_annual_metrics(d4, tariff_col)
    m4['scenario'] = 'Battery + Grid'; m4['tariff'] = tariff_name
    results.append(m4)

res_df = pd.DataFrame(results)
base_cost = res_df[res_df['scenario']=='Grid only'].set_index('tariff')['cost']
baseline_co2 = res_df[res_df['scenario']=='Grid only']['co2_kg'].iloc[0]  # same for all tariffs
res_df['savings'] = res_df.apply(lambda r: base_cost[r['tariff']] - r['cost'], axis=1)
res_df['co2_savings_kg'] = baseline_co2 - res_df['co2_kg']
res_df['co2_reduction_pct'] = 100 * res_df['co2_savings_kg'] / baseline_co2
# Sanity check: Battery+Grid should differ from Grid only (if same, report a bug)
grid_cost = res_df[res_df['scenario']=='Grid only']['cost'].iloc[0]
batt_cost = res_df[res_df['scenario']=='Battery + Grid']['cost'].iloc[0]
if abs(grid_cost - batt_cost) < 1:
    print('WARNING: Battery+Grid cost equals Grid only - battery logic may need checking')
res_df

---
## Step 6.5: Speed settings (optional – run this BEFORE Step 7 if optimizer is too slow)

**Change the numbers below** and run this cell. Then run Step 7.

In [ ]:
# ========== MAKE OPTIMIZER FASTER ==========
# Default (slow, best precision):  PV_STEP=1,   BATT_STEP=5
# Quick (~30 sec):                 PV_STEP=5,   BATT_STEP=10
# Very fast (~10 sec):             PV_STEP=10,  BATT_STEP=20

PV_STEP = 5
BATT_STEP = 10

print(f"Speed: PV step={PV_STEP}, Battery step={BATT_STEP}")

---
## Step 7: System optimizer

Tests PV and battery sizes for three configurations:
- **PV + Grid**
- **PV + Battery + Grid**
- **Battery + Grid**

Computes: annual cost, savings, payback, NPV, self-sufficiency, CO₂ savings.

**Too slow?** In the cell above, change `PV_STEP = 5` and `BATT_STEP = 10` for a quicker run (~30 sec). See README for details.

In [ ]:
PV_COST_PER_KWP = 1000   # €/kWp
BATT_COST_PER_KWH = 300  # €/kWh
DISCOUNT_RATE = 0.05
LIFETIME_YEARS = 20
TOTAL_CONSUMPTION = df['consumption'].sum()
BASELINE_COST = (df['consumption'] * df['tariff_standard']).sum()  # Grid-only cost (standard tariff)
baseline_co2 = (df['consumption'] * CO2_FACTOR).sum()

# Use speed from Step 6.5 if you ran it; else default (5,10=quick). Change (5),(10) here for fastest fix.
PV_STEP = globals().get('PV_STEP', 5)
BATT_STEP = globals().get('BATT_STEP', 10)

def optimize(tariff_col='tariff_standard'):
    base_cost = (df['consumption'] * df[tariff_col]).sum()  # Grid-only for this tariff
    rows = []
    # PV only: 10-150 kWp
    for pv in range(10, 151, PV_STEP):
        d = run_scenario_pv_grid(df, pv, tariff_col)
        cost = (d['grid_import'] * d[tariff_col]).sum() - (d['feed_in'] * EXPORT_RATE).sum()
        inv = pv * PV_COST_PER_KWP
        savings = base_cost - cost  # vs grid-only for this tariff
        ss = 100 * (1 - d['grid_import'].sum() / TOTAL_CONSUMPTION)
        co2_save = baseline_co2 - (d['grid_import'] * CO2_FACTOR).sum()
        payback = inv / savings if savings > 0 else np.inf
        npv = -inv + sum(savings / (1 + DISCOUNT_RATE)**t for t in range(1, LIFETIME_YEARS + 1))
        rows.append({'config': 'PV only', 'pv_kwp': pv, 'batt_kwh': 0, 'cost': cost, 'inv': inv, 'savings': savings, 'payback': payback, 'npv': npv, 'self_suff_pct': ss, 'co2_save_kg': co2_save})
    # Battery only: 0-300 kWh, step 5
    for batt in range(0, 301, BATT_STEP):
        if batt == 0:
            continue
        d = run_scenario_battery_grid(df, batt, tariff_col)
        cost = (d['grid_import'] * d[tariff_col]).sum()
        inv = batt * BATT_COST_PER_KWH
        savings = base_cost - cost
        ss = 100 * (1 - d['grid_import'].sum() / TOTAL_CONSUMPTION)
        co2_save = baseline_co2 - (d['grid_import'] * CO2_FACTOR).sum()
        payback = inv / savings if savings > 0 else np.inf
        npv = -inv + sum(savings / (1 + DISCOUNT_RATE)**t for t in range(1, LIFETIME_YEARS + 1))
        rows.append({'config': 'Battery only', 'pv_kwp': 0, 'batt_kwh': batt, 'cost': cost, 'inv': inv, 'savings': savings, 'payback': payback, 'npv': npv, 'self_suff_pct': ss, 'co2_save_kg': co2_save})
    # PV + Battery (batt >= 5 to avoid duplicating PV-only)
    for pv in range(10, 151, PV_STEP):
        for batt in range(5, 301, BATT_STEP):
            d = run_scenario_pv_battery_grid(df, pv, batt, tariff_col)
            cost = (d['grid_import'] * d[tariff_col]).sum() - (d['feed_in'] * EXPORT_RATE).sum()
            inv = pv * PV_COST_PER_KWP + batt * BATT_COST_PER_KWH
            savings = base_cost - cost
            ss = 100 * (1 - d['grid_import'].sum() / TOTAL_CONSUMPTION)
            co2_save = baseline_co2 - (d['grid_import'] * CO2_FACTOR).sum()
            payback = inv / savings if savings > 0 else np.inf
            npv = -inv + sum(savings / (1 + DISCOUNT_RATE)**t for t in range(1, LIFETIME_YEARS + 1))
            rows.append({'config': 'PV + Battery', 'pv_kwp': pv, 'batt_kwh': batt, 'cost': cost, 'inv': inv, 'savings': savings, 'payback': payback, 'npv': npv, 'self_suff_pct': ss, 'co2_save_kg': co2_save})
    return pd.DataFrame(rows)

print("Running optimizer (may take 1-2 minutes)...")
opt_df = optimize('tariff_standard')
opt_df.head(20)

---
## Step 8: Best configurations by goal

| Goal | Best config |
|------|-------------|
| Lowest bill | min cost |
| Highest savings | max savings |
| Best payback | min payback (finite) |
| Best self-sufficiency | max self_suff_pct |
| Most CO₂ saved | max co2_save_kg |
| Best NPV | max npv |

In [ ]:
# Filter out infinite payback for display
opt_f = opt_df[opt_df['payback'] < 100].copy()

goals = [
    ('Lowest annual cost', 'cost', 'idxmin'),
    ('Highest savings', 'savings', 'idxmax'),
    ('Best payback (years)', 'payback', 'idxmin'),
    ('Best self-sufficiency %', 'self_suff_pct', 'idxmax'),
    ('Highest CO2 savings (kg)', 'co2_save_kg', 'idxmax'),
    ('Best NPV (20y)', 'npv', 'idxmax'),
]
for name, col, fn in goals:
    idx = getattr(opt_df[col], fn)()
    r = opt_df.loc[idx]
    print(f"{name}: {r['config']} | PV={r['pv_kwp']} kWp, Batt={r['batt_kwh']} kWh | {col}={r[col]:.2f}")

# Best configs that include battery
with_batt = opt_df[opt_df['batt_kwh'] > 0]
if len(with_batt) > 0:
    print("\n--- Best WITH battery ---")
    r = with_batt.loc[with_batt['cost'].idxmin()]
    print(f"Lowest cost: PV={r['pv_kwp']} kWp, Batt={r['batt_kwh']} kWh")
    r = with_batt.loc[with_batt['npv'].idxmax()]
    print(f"Best NPV:    PV={r['pv_kwp']} kWp, Batt={r['batt_kwh']} kWh")

---
## Step 9: Charts (optional)

Simple visualizations of scenario comparison and optimizer results.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Scenario comparison
sns.barplot(data=res_df, x='scenario', y='cost', hue='tariff', ax=axes[0])
axes[0].set_title('Annual electricity cost by scenario')
axes[0].tick_params(axis='x', rotation=15)
# NPV by config type
for cfg in opt_df['config'].unique():
    sub = opt_df[opt_df['config']==cfg]
    axes[1].scatter(sub['pv_kwp'] + sub['batt_kwh']/50, sub['npv'], label=cfg, alpha=0.5, s=10)
axes[1].set_xlabel('PV kWp + Battery/50 (rough size)')
axes[1].set_ylabel('NPV (€)')
axes[1].set_title('NPV by configuration')
axes[1].legend()
plt.tight_layout()
plt.show()

---
## Summary

You have run:
1. **Data load** – consumption + PV profile (per kWp from Timeseries)
2. **Tariff setup** – Standard, Weekend Saver, Flat
3. **Four scenarios** – Grid only, PV + Grid, PV + Battery + Grid, Battery + Grid
4. **Optimizer** – best PV/battery sizes for multiple goals
5. **Metrics** – cost, savings, payback, NPV, CO₂

**To run for other tariffs:** change `tariff_col` in the optimizer call to `'tariff_weekend'` or `'tariff_flat'`.